# Becke Pruning Module Tests

This notebook contains comprehensive tests for the `molgrid.prune` module, specifically the `Becke` class for computing Becke weights for molecular grids.

In [3]:
import numpy as np
from molgrid import Molecule, Atom, Becke

# Test polynomial_fk with different iterations
mol = Molecule([Atom('O', [0.0, 0.0, 0.0])])
becke = Becke(mol)

# For k=1, fk should equal p
mu_test = 0.5
fk1 = becke._polynomial_fk(mu_test, k=1)
p1_check = becke._polynomial_p(mu_test)
assert np.isclose(fk1, p1_check), f"fk(mu, k=1) should equal p(mu)"

# For k=0, fk should equal mu (identity)
fk0 = becke._polynomial_fk(mu_test, k=0)
assert np.isclose(fk0, mu_test), f"fk(mu, k=0) should equal mu"

# For k=2, fk(mu, 2) = p(p(mu))
p_mu = becke._polynomial_p(mu_test)
p_p_mu = becke._polynomial_p(p_mu)
fk2 = becke._polynomial_fk(mu_test, k=2)
assert np.isclose(fk2, p_p_mu), f"fk(mu, k=2) should equal p(p(mu))"

# For k=3 (default)
fk3 = becke._polynomial_fk(mu_test, k=3)
fk3_manual = becke._polynomial_p(fk2)
assert np.isclose(fk3, fk3_manual), f"fk(mu, k=3) calculation is incorrect"

print("✓ Test 2 passed: Polynomial fk function (iteration) works correctly")
print(f"  fk({mu_test}, k=0) = {fk0}")
print(f"  fk({mu_test}, k=1) = {fk1}")
print(f"  fk({mu_test}, k=2) = {fk2}")
print(f"  fk({mu_test}, k=3) = {fk3}")

✓ Test 2 passed: Polynomial fk function (iteration) works correctly
  fk(0.5, k=0) = 0.5
  fk(0.5, k=1) = 0.6875
  fk(0.5, k=2) = 0.8687744140625
  fk(0.5, k=3) = 0.9752996308188813


In [4]:
# Test smoothing function bounds and properties
mol = Molecule([Atom('C', [0.0, 0.0, 0.0])])
becke = Becke(mol)

# Test at mu = 0: fk(0) = 0, so s(0) = 0.5
s0 = becke._smoothing_function(0.0)
assert np.isclose(s0, 0.5), f"Expected s(0) = 0.5, got {s0}"

# Test at mu = 1: fk(1) = 1, so s(1) = 0
s1 = becke._smoothing_function(1.0)
assert np.isclose(s1, 0.0), f"Expected s(1) = 0.0, got {s1}"

# Test at mu = -1: fk(-1) = -1, so s(-1) = 1.0
sm1 = becke._smoothing_function(-1.0)
assert np.isclose(sm1, 1.0), f"Expected s(-1) = 1.0, got {sm1}"

# Test smoothing function is bounded between 0 and 1
test_values = np.linspace(-1, 1, 11)
smoothed = [becke._smoothing_function(mu) for mu in test_values]
assert all(0 <= s <= 1 for s in smoothed), "Smoothing function should be bounded [0, 1]"

# Test with different k values
s_k1 = becke._smoothing_function(0.5, k=1)
s_k2 = becke._smoothing_function(0.5, k=2)
s_k3 = becke._smoothing_function(0.5, k=3)
assert 0 <= s_k1 <= 1 and 0 <= s_k2 <= 1 and 0 <= s_k3 <= 1
print("✓ Test 3 passed: Smoothing function works correctly")
print(f"  s(0) = {s0}")
print(f"  s(1) = {s1}")
print(f"  s(-1) = {sm1}")
print(f"  s(0.5, k=1) = {s_k1}")
print(f"  s(0.5, k=3) = {s_k3}")

✓ Test 3 passed: Smoothing function works correctly
  s(0) = 0.5
  s(1) = 0.0
  s(-1) = 1.0
  s(0.5, k=1) = 0.15625
  s(0.5, k=3) = 0.012350184590559365


In [5]:
# Test with very close atoms
mol_close = Molecule([
    Atom('He', [0.0, 0.0, 0.0]),
    Atom('He', [0.001, 0.0, 0.0])  # Very close
])
becke_close = Becke(mol_close)

# Should still give normalized weights
weights_close = becke_close._weight_function(np.array([0.0005, 0.0, 0.0]))
assert np.isclose(sum(weights_close), 1.0), "Weights should normalize even for close atoms"
assert all(0 <= w <= 1 for w in weights_close), "Weights should be in [0,1]"

# Test with very far atoms
mol_far = Molecule([
    Atom('Ar', [0.0, 0.0, 0.0]),
    Atom('Ar', [1000.0, 0.0, 0.0])
])
becke_far = Becke(mol_far)

# Should still give normalized weights
test_point = np.array([500.0, 0.0, 0.0])
weights_far = becke_far._weight_function(test_point)
assert np.isclose(sum(weights_far), 1.0), "Weights should normalize even for far atoms"
assert all(0 <= w <= 1 for w in weights_far), "Weights should be in [0,1]"

# Test with realistic mu values (bounded between -1 and 1 by definition)
# In Becke scheme: mu = (rip - rjp) / rij is always in [-1, 1] for physical coordinates
test_mu_values = [-1.0, -0.8, -0.5, -0.2, 0.0, 0.2, 0.5, 0.8, 1.0]
mol_test = Molecule([Atom('C', [0.0, 0.0, 0.0])])
becke_test = Becke(mol_test)

for mu in test_mu_values:
    p_val = becke_test._polynomial_p(mu)
    s_val = becke_test._smoothing_function(mu)
    assert not np.isnan(p_val) and not np.isinf(p_val), f"p({mu}) is NaN or Inf"
    assert not np.isnan(s_val) and not np.isinf(s_val), f"s({mu}) is NaN or Inf"
    assert 0 <= s_val <= 1, f"Smoothing value {s_val} outside [0,1] for mu={mu}"

print("✓ Test 8 passed: Numerical stability confirmed")
print(f"  Close atoms (distance 0.001): w1={weights_close[0]:.4f}, w2={weights_close[1]:.4f}")
print(f"  Far atoms (distance 1000): w1={weights_far[0]:.6f}, w2={weights_far[1]:.6f}")

✓ Test 8 passed: Numerical stability confirmed
  Close atoms (distance 0.001): w1=0.5000, w2=0.5000
  Far atoms (distance 1000): w1=0.500000, w2=0.500000


In [6]:
print("\n" + "="*60)
print("BECKE PRUNING MODULE - TEST SUMMARY")
print("="*60)
print("✓ Test 1: Polynomial p function")
print("✓ Test 2: Polynomial fk function (iteration)")
print("✓ Test 3: Smoothing function")
print("✓ Test 4: Single atom molecule")
print("✓ Test 5: H2 diatomic molecule")
print("✓ Test 6: H2O triatomic molecule")
print("✓ Test 7: Voronoi polyhedron")
print("✓ Test 8: Numerical stability")
print("="*60)
print("All tests passed! ✓")
print("="*60)


BECKE PRUNING MODULE - TEST SUMMARY
✓ Test 1: Polynomial p function
✓ Test 2: Polynomial fk function (iteration)
✓ Test 3: Smoothing function
✓ Test 4: Single atom molecule
✓ Test 5: H2 diatomic molecule
✓ Test 6: H2O triatomic molecule
✓ Test 7: Voronoi polyhedron
✓ Test 8: Numerical stability
All tests passed! ✓


## Test Summary

All tests completed successfully! The Becke pruning module passes all checks.

## Test 8: Numerical Stability

Test numerical stability with extreme values and edge cases

In [7]:
# Test Voronoi polyhedron for single atom (should always be 1 since no neighbor)
mol = Molecule([Atom('N', [0.0, 0.0, 0.0])])
becke = Becke(mol)
atom_0 = mol[0]

# For single atom, voronoi polyhedron should be 1 (empty product)
voronoi_single = becke._voronoi_polyhedron(atom_0, np.array([1.0, 2.0, 3.0]))
assert np.isclose(voronoi_single, 1.0), f"Single atom voronoi should be 1.0, got {voronoi_single}"

# Test with two atoms
mol_two = Molecule([
    Atom('H', [0.0, 0.0, 0.0]),
    Atom('H', [2.0, 0.0, 0.0])
])
becke_two = Becke(mol_two)
atom_a = mol_two[0]
atom_b = mol_two[1]

# At the location of atom A
voronoi_at_a = becke_two._voronoi_polyhedron(atom_a, np.array([0.0, 0.0, 0.0]))
assert 0 <= voronoi_at_a <= 1.0, f"Voronoi should be between 0 and 1, got {voronoi_at_a}"

# At the center
center = np.array([1.0, 0.0, 0.0])
voronoi_center_a = becke_two._voronoi_polyhedron(atom_a, center)
voronoi_center_b = becke_two._voronoi_polyhedron(atom_b, center)
assert 0 <= voronoi_center_a <= 1.0, f"Voronoi should be between 0 and 1"
assert 0 <= voronoi_center_b <= 1.0, f"Voronoi should be between 0 and 1"
# Due to symmetry, they should be similar
assert np.isclose(voronoi_center_a, voronoi_center_b, rtol=0.1), "Voronoi values should be similar at center"

# Test with hetero disabled
voronoi_no_hetero = becke_two._voronoi_polyhedron(atom_a, center, do_becke_hetero=False)
assert 0 <= voronoi_no_hetero <= 1.0, f"Voronoi should be between 0 and 1"

print("✓ Test 7 passed: Voronoi polyhedron function works correctly")
print(f"  Single atom voronoi: {voronoi_single}")
print(f"  Two atoms at center - atom A: {voronoi_center_a:.4f}, atom B: {voronoi_center_b:.4f}")

✓ Test 7 passed: Voronoi polyhedron function works correctly
  Single atom voronoi: 1.0
  Two atoms at center - atom A: 0.5000, atom B: 0.5000


## Test 7: Voronoi Polyhedron

Test the Voronoi polyhedron function directly

In [8]:
# Test with H2O molecule
# Simple geometry: O at origin, H atoms at +x and at 45 degrees
mol_h2o = Molecule([
    Atom('O', [0.0, 0.0, 0.0]),
    Atom('H', [0.96, 0.0, 0.0]),
    Atom('H', [0.24, 0.93, 0.0])
])
becke_h2o = Becke(mol_h2o)

# Test at oxygen location
weights_at_o = becke_h2o._weight_function(np.array([0.0, 0.0, 0.0]))
assert len(weights_at_o) == 3, "Should have 3 atom weights"
assert np.isclose(sum(weights_at_o), 1.0), f"Weights should sum to 1.0, got {sum(weights_at_o)}"
assert weights_at_o[0] > weights_at_o[1], "Weight at O should be largest at O location"
assert weights_at_o[0] > weights_at_o[2], "Weight at O should be largest at O location"

# Test at H1 location
weights_at_h1 = becke_h2o._weight_function(np.array([0.96, 0.0, 0.0]))
assert np.isclose(sum(weights_at_h1), 1.0), f"Weights should sum to 1.0, got {sum(weights_at_h1)}"
assert weights_at_h1[1] > weights_at_h1[0], "Weight at H1 should be largest at H1 location"

# Test at midpoint between O and H1
midpoint = np.array([0.48, 0.0, 0.0])
weights_mid = becke_h2o._weight_function(midpoint)
assert np.isclose(sum(weights_mid), 1.0), f"Weights should sum to 1.0, got {sum(weights_mid)}"

# Test with heteroatomic correction disabled
weights_no_hetero = becke_h2o._weight_function(midpoint, do_becke_hetero=False)
assert np.isclose(sum(weights_no_hetero), 1.0), f"Weights should sum to 1.0 without hetero, got {sum(weights_no_hetero)}"

# Hetero correction should give different results
weights_with_hetero = becke_h2o._weight_function(midpoint, do_becke_hetero=True)
# Results may differ slightly due to heteroatomic correction
assert len(weights_with_hetero) == 3, "Should still have 3 weights with hetero correction"

print("✓ Test 6 passed: Becke weights for H2O molecule work correctly")
print(f"  At O: w_O={weights_at_o[0]:.4f}, w_H1={weights_at_o[1]:.4f}, w_H2={weights_at_o[2]:.4f}")
print(f"  At H1: w_O={weights_at_h1[0]:.4f}, w_H1={weights_at_h1[1]:.4f}, w_H2={weights_at_h1[2]:.4f}")
print(f"  At midpoint: w_O={weights_mid[0]:.4f}, w_H1={weights_mid[1]:.4f}, w_H2={weights_mid[2]:.4f}")

✓ Test 6 passed: Becke weights for H2O molecule work correctly
  At O: w_O=1.0000, w_H1=0.0000, w_H2=0.0000
  At H1: w_O=0.0000, w_H1=1.0000, w_H2=0.0000
  At midpoint: w_O=0.8918, w_H1=0.1082, w_H2=0.0000


## Test 6: Water Molecule (3-body System)

Test Becke weights for a water molecule (O, H, H)

In [9]:
# Test with H2 molecule: H at (0,0,0) and (1,0,0)
mol_h2 = Molecule([
    Atom('H', [0.0, 0.0, 0.0]),
    Atom('H', [1.0, 0.0, 0.0])
])
becke_h2 = Becke(mol_h2)

# Test at center (0.5, 0, 0) - should have equal weights due to symmetry
center_point = np.array([0.5, 0.0, 0.0])
weights_center = becke_h2._weight_function(center_point)
assert len(weights_center) == 2, "Should have 2 atom weights"
assert np.isclose(weights_center[0], weights_center[1]), "Weights should be equal at center due to symmetry"
assert np.isclose(sum(weights_center), 1.0), f"Weights should sum to 1.0, got {sum(weights_center)}"

# Test at atom 1 location - should be closer to atom 1
point_near_a1 = np.array([0.1, 0.0, 0.0])
weights_near_a1 = becke_h2._weight_function(point_near_a1)
assert weights_near_a1[0] > weights_near_a1[1], "Weight at atom 1 should be larger near atom 1"
assert np.isclose(sum(weights_near_a1), 1.0), f"Weights should sum to 1.0, got {sum(weights_near_a1)}"

# Test at atom 2 location - should be closer to atom 2
point_near_a2 = np.array([0.9, 0.0, 0.0])
weights_near_a2 = becke_h2._weight_function(point_near_a2)
assert weights_near_a2[1] > weights_near_a2[0], "Weight at atom 2 should be larger near atom 2"
assert np.isclose(sum(weights_near_a2), 1.0), f"Weights should sum to 1.0, got {sum(weights_near_a2)}"

# Test far from both atoms
far_point = np.array([100.0, 0.0, 0.0])
weights_far = becke_h2._weight_function(far_point)
assert np.isclose(sum(weights_far), 1.0), f"Weights should sum to 1.0, got {sum(weights_far)}"

print("✓ Test 5 passed: Becke weights for H2 molecule work correctly")
print(f"  Center (0.5,0,0): w1={weights_center[0]:.4f}, w2={weights_center[1]:.4f}")
print(f"  Near atom 1 (0.1,0,0): w1={weights_near_a1[0]:.4f}, w2={weights_near_a1[1]:.4f}")
print(f"  Near atom 2 (0.9,0,0): w1={weights_near_a2[0]:.4f}, w2={weights_near_a2[1]:.4f}")

✓ Test 5 passed: Becke weights for H2 molecule work correctly
  Center (0.5,0,0): w1=0.5000, w2=0.5000
  Near atom 1 (0.1,0,0): w1=1.0000, w2=0.0000
  Near atom 2 (0.9,0,0): w1=0.0000, w2=1.0000


## Test 5: Two Atom Molecule

Test Becke weights for a diatomic molecule (H2)

In [10]:
# Test with single atom
mol_single = Molecule([Atom('H', [0.0, 0.0, 0.0])])
becke_single = Becke(mol_single)

# Test at origin
weights_origin = becke_single._weight_function(np.array([0.0, 0.0, 0.0]))
assert len(weights_origin) == 1, "Should have 1 atom weight"
assert np.isclose(weights_origin[0], 1.0), f"Single atom weight should be 1.0, got {weights_origin[0]}"

# Test at arbitrary point
weights_arb = becke_single._weight_function(np.array([1.0, 2.0, 3.0]))
assert len(weights_arb) == 1, "Should have 1 atom weight"
assert np.isclose(weights_arb[0], 1.0), f"Single atom weight should be 1.0, got {weights_arb[0]}"

# Test at multiple points
test_points = [
    np.array([0.0, 0.0, 0.0]),
    np.array([1.0, 0.0, 0.0]),
    np.array([5.0, 5.0, 5.0]),
    np.array([-1.0, -1.0, -1.0])
]

for pt in test_points:
    weights = becke_single._weight_function(pt)
    assert np.isclose(weights[0], 1.0), f"Weight should be 1.0 at {pt}, got {weights[0]}"

print("✓ Test 4 passed: Single atom molecules have uniform weight 1.0")

✓ Test 4 passed: Single atom molecules have uniform weight 1.0


## Test 4: Single Atom Molecule

Test that a single atom molecule has weight 1.0 everywhere

## Test 3: Smoothing Function

Test the smoothing function: s(mu) = 0.5 * (1 - fk(mu))

## Test 2: Polynomial fk Function

Test the iterative polynomial function fk(mu) which applies p() k times

In [11]:
# Test polynomial_p at known values
mol = Molecule([Atom('H', [0.0, 0.0, 0.0])])
becke = Becke(mol)

# Test at mu = 0
p0 = becke._polynomial_p(0.0)
assert np.isclose(p0, 0.0), f"Expected p(0) = 0, got {p0}"

# Test at mu = 1
p1 = becke._polynomial_p(1.0)
expected_p1 = 1.5 * 1.0 - 0.5 * 1.0**3  # = 1.5 - 0.5 = 1.0
assert np.isclose(p1, expected_p1), f"Expected p(1) = 1.0, got {p1}"

# Test at mu = -1
pm1 = becke._polynomial_p(-1.0)
expected_pm1 = 1.5 * (-1.0) - 0.5 * (-1.0)**3  # = -1.5 + 0.5 = -1.0
assert np.isclose(pm1, expected_pm1), f"Expected p(-1) = -1.0, got {pm1}"

# Test at mu = 0.5
p05 = becke._polynomial_p(0.5)
expected_p05 = 1.5 * 0.5 - 0.5 * 0.5**3  # = 0.75 - 0.0625 = 0.6875
assert np.isclose(p05, expected_p05), f"Expected p(0.5) = {expected_p05}, got {p05}"

print("✓ Test 1 passed: Polynomial p function works correctly")
print(f"  p(0) = {p0}")
print(f"  p(1) = {p1}")
print(f"  p(-1) = {pm1}")
print(f"  p(0.5) = {p05}")

✓ Test 1 passed: Polynomial p function works correctly
  p(0) = 0.0
  p(1) = 1.0
  p(-1) = -1.0
  p(0.5) = 0.6875


## Test 1: Polynomial p Function

Test the basic polynomial: p(mu) = 1.5*mu - 0.5*mu³

In [12]:
# Import required modules
import numpy as np
import sys
sys.path.insert(0, '/Users/yilin/Documents/GitHub/molgrid')

from molgrid.molecule import Atom, Molecule
from molgrid.prune import Becke
from molgrid.moleculargrid import MolecularGrid

print("✓ Imports successful")

✓ Imports successful
